**Bloque 01. IPython y Jupyter**

Revisión y práctica del *Python Data Science Handbook*, de Jake VanderPlas.
Este cuaderno contiene explicaciones y ejemplos propios en español.

El objetivo es construir un flujo de exploración: consultar ayuda, ejecutar código, revisar el historial, interpretar un error y medir una operación.

Requisitos: Python 3, Jupyter e IPython; NumPy para la comparación de tiempos. Ejecuta las celdas en orden desde un kernel nuevo.
Los datos de los ejemplos son sintéticos, salvo que se indique otra procedencia.
Cada bloque es independiente de los demás.

**1. El notebook como registro de trabajo**

Python ejecuta el cálculo; el kernel de IPython agrega herramientas interactivas.
Jupyter organiza texto, código y resultados en celdas. El orden de ejecución puede
diferir del orden visual: reiniciar el kernel y ejecutar todo ayuda a detectar
dependencias ocultas. Las versiones se muestran para documentar el entorno.

In [1]:
import sys
import inspect
import traceback
from pathlib import Path
from tempfile import TemporaryDirectory

import IPython
import numpy as np

print('Python:', sys.version.split()[0])
print('IPython:', IPython.__version__)
print('NumPy:', np.__version__)

Python: 3.12.10
IPython: 9.17.1
NumPy: 2.3.5


**2. Ayuda, código fuente y autocompletado**

`objeto?` muestra información de un objeto y `objeto??` intenta mostrar también
su implementación. `help()` funciona en Python normal. Para explorar nombres,
escribe `np.` y pulsa Tab; los atajos dependen de la interfaz. En Jupyter,
Shift+Enter suele ejecutar la celda. Los atajos de la terminal de IPython del
libro se consultan en su sección original y no todos se aplican al navegador.

In [2]:
def media(valores):
    """Calcula la media de una secuencia no vacía de números."""
    if len(valores) == 0:
        raise ValueError('Se necesita al menos un dato.')
    return sum(valores) / len(valores)

help(media)
print('Firma:', inspect.signature(media))
print('Nombres de NumPy relacionados con medias:',
      [nombre for nombre in dir(np) if 'mean' in nombre][:8])

Help on function media in module __main__:

media(valores)
    Calcula la media de una secuencia no vacía de números.

Firma: (valores)
Nombres de NumPy relacionados con medias: ['mean', 'nanmean']


In [3]:
media?

In [4]:
media??

**3. Comandos mágicos de línea y de celda**

Un comando con `%` actúa sobre una línea; uno con `%%` recibe toda la celda.
Son herramientas del kernel de IPython. `%pwd` consulta la carpeta de trabajo
y `%%time` informa del tiempo de ejecución de esta celda concreta.

In [5]:
%pwd

'C:\\Users\\Aldo\\Desktop\\ICD\\Notebooks'

In [6]:
%%time
mediciones = [8, 12, 15, 9, 11, 17]
media_mediciones = media(mediciones)
print('Media de las mediciones sintéticas:', media_mediciones)

Media de las mediciones sintéticas: 12.0
CPU times: total: 0 ns
Wall time: 0 ns


**4. Historial y salida**

`In` conserva entradas de la sesión y `Out` almacena resultados de expresiones.
Una salida de `print()` no se guarda como resultado en `Out`. El punto y coma
al final de una expresión suprime su presentación automática. `%history`
ayuda a recuperar comandos, pero el notebook debe conservar el código necesario
para reproducir el análisis sin depender del historial de otra sesión.

In [7]:
sum(mediciones)

72

In [8]:
print('Entradas en esta sesión:', len(In))
print('Índices con resultados:', sorted(Out))
%history -n -l 3

Entradas en esta sesión: 9
Índices con resultados: [5, 7]
   5: %pwd
   6:
%%time
mediciones = [8, 12, 15, 9, 11, 17]
media_mediciones = media(mediciones)
print('Media de las mediciones sintéticas:', media_mediciones)
   7: sum(mediciones)


In [9]:
sum(mediciones);

**5. Interacción con el sistema y ejecución de un script**

`!` envía una orden al shell; su sintaxis depende del sistema operativo.
El ejemplo usa `echo`. Para trabajar con rutas de forma portable uso `pathlib`.
`%run` permite ejecutar un archivo Python. Aquí el archivo se crea en una
carpeta temporal que se elimina automáticamente al terminar.

In [10]:
!echo Ejemplo de una orden del sistema desde IPython

Ejemplo de una orden del sistema desde IPython


In [11]:
with TemporaryDirectory() as carpeta:
    ruta_script = Path(carpeta) / 'resumen.py'
    ruta_script.write_text(
        "datos_script = [4, 6, 8]\nprint('Total del script:', sum(datos_script))\n",
        encoding='utf-8',
    )
    get_ipython().run_line_magic('run', f'"{ruta_script}"')

Total del script: 18


**6. Errores y depuración**

Un traceback indica el recorrido que llevó a la excepción. Provoco y capturo
un error de entrada para revisar su mensaje sin interrumpir las demás celdas.
La solución es comprobar el caso vacío antes de calcular la media, como hace
la función. Para investigar una excepción sin capturar se puede ejecutar
`%debug` manualmente después del fallo; ese depurador es interactivo.

In [12]:
try:
    media([])
except ValueError:
    print(traceback.format_exc(limit=1))

assert media([4, 6, 8]) == 6

Traceback (most recent call last):
  File "D:\Temp\ipykernel_4156\326735429.py", line 2, in <module>
    media([])
ValueError: Se necesita al menos un dato.



**7. Medición y perfilado**

`%timeit` repite un cálculo para estimar su duración; `%prun` resume llamadas
y tiempos de funciones. Comparo dos formas de elevar al cuadrado los mismos
números. La creación de los datos queda fuera de la medición. Los tiempos
dependen de la máquina y no sirven como una promesa general de rendimiento.
Las extensiones `%lprun` y `%memit` del libro requieren paquetes adicionales;
el ejemplo ejecutable se limita a herramientas incluidas con IPython.

In [13]:
lista = list(range(20_000))
arreglo = np.arange(20_000, dtype=np.int64)
assert np.array_equal(np.array([x * x for x in lista]), arreglo * arreglo)
%timeit -r 3 -n 30 [x * x for x in lista]
%timeit -r 3 -n 30 arreglo * arreglo

1.4 ms ± 13.4 μs per loop (mean ± std. dev. of 3 runs, 30 loops each)


35.2 μs ± 11.1 μs per loop (mean ± std. dev. of 3 runs, 30 loops each)


In [14]:
%prun -s cumulative -l 8 sum(x * x for x in range(20_000))

**8. Ejercicio resuelto y conclusión**

Pregunta: ¿cómo resumir una secuencia y detectar una entrada inválida?
La solución reutiliza una función documentada y devuelve el número de datos,
la media y los extremos. Separar esa función de las herramientas interactivas
permite reutilizar el cálculo en otros programas.

In [15]:
def resumir(valores):
    """Devuelve cantidad, media, mínimo y máximo de una secuencia no vacía."""
    promedio = media(valores)
    return {'n': len(valores), 'media': promedio,
            'minimo': min(valores), 'maximo': max(valores)}

resumen = resumir(mediciones)
assert resumen['n'] == 6
assert resumen['minimo'] == 8
print(resumen)

{'n': 6, 'media': 12.0, 'minimo': 8, 'maximo': 17}


El resultado integra cálculo, documentación y verificación en el mismo archivo.
La ayuda permite explorar la API; el historial facilita la exploración;
el manejo de excepciones explica un fallo; las mediciones orientan una mejora.
Para compartirlo conviene guardar las salidas y comprobar que todas las celdas
se pueden ejecutar desde un kernel nuevo.

**Fuentes y correspondencia con el libro**

Referencia indicada en la consigna: VanderPlas, J. (2022). *Python Data Science Handbook: Essential Tools for Working with Data*. O’Reilly Media.

El material efectivamente revisado es la carpeta `notebooks_v1` del enlace de la consigna. Su prefacio identifica el copyright de 2016; por eso la referencia de 2022 y estos materiales no se presentan como una misma edición. Las secciones siguientes permiten consultar el desarrollo original de los temas practicados aquí.

[01.00-IPython-Beyond-Normal-Python](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.00-IPython-Beyond-Normal-Python.ipynb).

[01.01-Help-And-Documentation](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.01-Help-And-Documentation.ipynb).

[01.02-Shell-Keyboard-Shortcuts](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.02-Shell-Keyboard-Shortcuts.ipynb).

[01.03-Magic-Commands](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.03-Magic-Commands.ipynb).

[01.04-Input-Output-History](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.04-Input-Output-History.ipynb).

[01.05-IPython-And-Shell-Commands](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.05-IPython-And-Shell-Commands.ipynb).

[01.06-Errors-and-Debugging](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.06-Errors-and-Debugging.ipynb).

[01.07-Timing-and-Profiling](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.07-Timing-and-Profiling.ipynb).

[01.08-More-IPython-Resources](https://github.com/husseinlopez/PythonDataScienceHandbook/blob/master/notebooks_v1/01.08-More-IPython-Resources.ipynb).

Documentación de las bibliotecas utilizada para adaptar los ejemplos:

[Comandos mágicos de IPython](https://ipython.readthedocs.io/en/stable/interactive/magics.html).

El texto original del libro declara licencia CC BY-NC-ND y su código, MIT. Las explicaciones y los ejercicios de este cuaderno son una elaboración nueva; las fuentes se conservan como referencia de estudio.